# 06 · Solutions — RWA RealT replication (Lab 4.8)

Worked answers to the four exercises in `06_rwa_realt_replication.ipynb`.

> **Attempt the exercises in notebook 06 first** — open this only to check your work.


## Reference answers

1. **Ownership stability.** As the sample widens, the medians stay close to the headline: about **142 real holders**, **HHI\* ≈ 1.9%**, and **~75% of supply in contracts**. Concentration stays low, so the dispersed-ownership finding is robust across the RealT universe.
2. **Turnover.** For 15777 Ardmore, recent monthly turnover is about **0.3% (median), though some 2025 months reach about 5–7%** — *far below* Swinkels' 15%→5%. This refines the lecture's point: the move to Gnosis added the *infrastructure* for liquidity (a cheaper chain and AMM pools), but this turnover shows that actual trading against it is still minimal — the gap is addressed in **structure, not yet in practice**; cheaper gas alone did not restore trading. Computed from the cached explorer transfers in the next cell.
3. **Contract holders.** The ~75% is held by the Gnosis **AMB bridge mediator** (tokens bridged from Ethereum) and the **AMM/DEX liquidity pools**. Filtering `is_contract` holders and summing their balances reproduces the ~75% share.
4. **Detroit house-price index (FRED `DEXRSA`).** Jan 2019 ≈ **125.2** → Feb 2021 ≈ **145.4** (**+16%** over Swinkels' window) → May 2026 ≈ **201.5** (**+61%** since Jan 2019). Detroit prices rose substantially; the paper found token prices tracked this trend. (Replicating that tracking on-chain today needs DEX trade prices, since RealT's quoted `tokenPrice` is a fixed primary price.)


In [1]:
import json, collections
import pandas as pd
# Reference for task 2: monthly turnover for 15777 Ardmore from cached free explorer /transfers
items = json.load(open('data/rwa/transfers_ardmore.json'))   # free Gnosis Blockscout /transfers
SUPPLY = 1300.0
permonth = collections.defaultdict(float)
for it in items:
    ts = it.get('timestamp') or ''
    tot = it.get('total') or {}
    v = tot.get('value'); dec = int(tot.get('decimals') or 18)
    if ts and v is not None:
        permonth[ts[:7]] += int(v) / (10 ** dec)
tt = pd.DataFrame(sorted((m, permonth[m], 100*permonth[m]/SUPPLY) for m in permonth),
                  columns=['month', 'tokens_moved', 'turnover_%'])
print('Monthly turnover for 15777 Ardmore (upper bound; includes bridge/AMM moves):')
print(tt.tail(8).to_string(index=False))
print(f"\nMedian monthly turnover, last 6 months: {tt['turnover_%'].tail(6).median():.1f}%   (Swinkels 2023: 15% -> 5%)")


Monthly turnover for 15777 Ardmore (upper bound; includes bridge/AMM moves):
  month  tokens_moved   turnover_%
2025-12     21.050001 1.619231e+00
2026-01     45.443279 3.495637e+00
2026-02      6.882090 5.293915e-01
2026-03      6.153955 4.733812e-01
2026-05      4.000000 3.076923e-01
2026-06      1.000000 7.692308e-02
2026-07      0.000010 7.692308e-07
2026-08      0.001000 7.692308e-05

Median monthly turnover, last 6 months: 0.2%   (Swinkels 2023: 15% -> 5%)
